In [1]:
import pandas as pd
import json
import regex as re
import html
import zipfile
import os

## Create lists of URLs and domains

In [52]:
def create_final_lists(corpus, file_path):
	with open(file_path, "r") as cur_file:
		urls = json.load(cur_file)

	print("Number of URLs: ")
	print(len(urls))
	print(urls[:10])

	with open(f"final/CLASSLA-web{corpus}.urls.json", "w", encoding="utf-8") as new_url_file:
		json.dump(sorted(urls), new_url_file, ensure_ascii=False,indent=4)


	print(f"New URL list saved as: final/CLASSLA-web{corpus}.urls.json")

	# Create a list of domains from that
	domain_re=re.compile(r'^https?://(?:www\.)?(.+?)[/$]')

	domains = []

	for curr_url in urls:
		curr_domain=html.escape(domain_re.search(curr_url).group(1))
		domains.append(curr_domain)

	print("Number of unique domains: ")

	unique_domains = sorted(list(set(domains)))

	print(len(unique_domains))

	print(unique_domains[:10])

	with open(f"final/CLASSLA-web{corpus}.domains.json", "w", encoding="utf-8") as new_domain_file:
		json.dump(unique_domains, new_domain_file, ensure_ascii=False,indent=4)

	print(f"New list of unique domains saved as: final/CLASSLA-web{corpus}.domains.json")




In [53]:
corpus = ".sl.1.0"

file_path = "CLASSLA-web-sl.1.0-urls.json-url-list.json"

In [54]:
create_final_lists(corpus, file_path)

Number of URLs: 
4063462
['https://slo-tech.com/novice/t708501', 'https://govorise.metropolitan.si/traci/domaci-traci/tanja-ribic-pokazala-kako-zelo-spretna-je-za-volanom-svoje-jahte/', 'https://www.lopa.si/profi-sistem-y-kos.html', 'https://www.merkur.si/akuml-kosilnica-z-nitko-worx-20v-1x2-0-ah-baterija-in-polnilnik/', 'https://www.editor.si/na-editorju-sirimo-ekipo', 'http://www.zkdl.si/index.php?option=com_content&amp;amp;view=article&amp;amp;id=595:slovenski-glasbeni-dnevi&amp;amp;catid=88:arhiv&amp;amp;Itemid=110', 'https://www.gigaspark.com/knowledgebase/41/Kaksen-je-postopek-za-prenos-domene-na-Gigaspark.html', 'https://regionalobala.si/novica/tri-mesece-je-zivel-na-letaliscu-in-nihce-tega-ni-vedel-zaradi-strahu-pred-koronavirusom-si-ni-upal-', 'http://poslovni-imenik.si/toplozracni-kamini/', 'https://www.vecer.com/slovenija/prevzem-pro-plusa-kaj-bo-danes-odlocil-senat-avk-6435647']
New URL list saved as: final/CLASSLA-web.sl.1.0.urls.json
Number of unique domains: 
49652
['057

## Create blacklists

In [12]:
blacklist = []

In [37]:
# First, add all "blacklist.domains" lists

path = "si_blacklist.domains" 

In [38]:
with open(path, "r") as blacklist_path:
	b_list = blacklist_path.readlines()

b_list = [x.strip("\n") for x in b_list]

b_list[:10]

['your.eye.rsiurnik.si',
 'kawaii.cute.si',
 'nepi.si',
 'digital.boot.si',
 'gdri.si',
 'indy.go-tisk.si',
 'googlebot.si',
 'perfekt.si',
 'server.lost.si',
 'email.si']

In [39]:
for el in b_list:
	blacklist.append(el)

len(blacklist)

367

In [40]:
with open(f"final/CLASSLA-web.blacklisted_domains.json", "w", encoding="utf-8") as new_blacklist_file:
	json.dump(blacklist, new_blacklist_file, ensure_ascii=False,indent=4)

Open the manually-evaluated files to add bad domains to the blacklist

In [47]:
addition = pd.read_csv("final/CLASSLA-web-bad-domains-more-information.txt", sep="\t", index_col = 0)
addition.head(2)

,domain,comment,source_corpus
0,machineseeker.hr,machine translation,MaCoCu-hr 1.0
1,mi.warbletoncouncil.org,foreign languages,MaCoCu-hr 1.0


In [48]:
addition["source_corpus"].unique()

array(['MaCoCu-hr 1.0', 'MaCoCu-sl 1.0', 'MaCoCu HBS corpora 2.0',
       'CLASSLA-web 2.0 (Western Slavic)',
       'CLASSLA-web 2.0 (Eastern Slavic)'], dtype=object)

In [43]:
addition.shape

(497, 3)

In [42]:
add_domains = addition["domain"].to_list()
add_domains[:4]

['machineseeker.hr',
 'mi.warbletoncouncil.org',
 'haw.warbletoncouncil.org',
 'bs.warbletoncouncil.org']

In [44]:
for el in add_domains:
	blacklist.append(el)

len(blacklist)

864

In [55]:
# Edit the list - remove duplicates and sort it alphabetically

final_blacklist = sorted(list(set(blacklist)))

print(len(final_blacklist))

final_blacklist[:10]

682


['1001beach.com',
 '11307.a.hostable.me',
 '41ini.me',
 '43ole.me',
 '4import.me',
 '55pif.me',
 '5volt.eu',
 '97type.me',
 'abv.bg',
 'actualno.com']

In [57]:
with open(f"final/CLASSLA-web.blacklisted_domains.json", "w", encoding="utf-8") as new_blacklist_file:
	json.dump(final_blacklist, new_blacklist_file, ensure_ascii=False,indent=4)

In [2]:
with open(f"final/CLASSLA-web.blacklisted_domains.json", "r") as new_blacklist_file:
	blacklist = json.load(new_blacklist_file)

blacklist[:2]

['1001beach.com', '11307.a.hostable.me']

In [3]:
len(blacklist)

682

## Create final files providing information on duplicated texts

In [50]:
def create_dup_file(path, suffix):
	df = pd.read_json(path)

	df.rename(columns={0: "id", 1: "duplicate"}, inplace=True)

	print(df.shape)

	display(df.head(2))

	# Keep only duplicates
	df = df[df["duplicate"] != "no"]

	print("Keep only duplicates. New size: ")
	print(df.shape)

	# Rename all ids
	new_ids = []

	new_ids = [x.replace(f"-2024.{suffix}", f".2.0.{suffix}") for x in df["id"].to_list()]

	df["id"] = new_ids

	df.set_index("id", inplace=True)

	display(df.head(2))

	final_dict = df.to_dict()["duplicate"]

	with open(f"CLASSLA-web.{suffix}.2.0.duplicates_with_CLASSLA-web.{suffix}.1.0.json", "w") as duplicate_file:
		json.dump(final_dict, duplicate_file, indent=2)

	print(f"File saved as CLASSLA-web.{suffix}.2.0.duplicates_with_CLASSLA-web.{suffix}.1.0.json")

In [58]:
suffix = "bg"
path = "near-duplicates-in-CLASSLA-web-2024.bg-from-CLASSLA-web-bg.1.0.json"

In [59]:
create_dup_file(path, suffix)

(14668969, 2)


,id,duplicate
0,CLASSLA-web-2024.bg.1,no
1,CLASSLA-web-2024.bg.2,no


Keep only duplicates. New size: 
(2084701, 2)


,duplicate
id,
CLASSLA-web.2.0.bg.9,[CLASSLA-web.bg.1277]
CLASSLA-web.2.0.bg.25,[CLASSLA-web.bg.2191]


File saved as CLASSLA-web.bg.2.0.duplicates_with_CLASSLA-web.bg.1.0.json
